# Step 2 — Synthetic Patient Generator (V1)

This notebook builds the **first synthetic longitudinal multimodal dataset** for:

**Bioprosthetic valve durability prediction**

It uses the existing `multimodal_patient_year.csv` only as a **reference** for:
- existing feature names,
- empirical lab distributions,
- medication prevalences,
- missingness patterns,
- patient-year structure.

It then adds valve-specific variables that are missing from the current dataset.

> **Important:** valve-specific hemodynamic ranges and deterioration rules in this notebook are **illustrative model assumptions** until they are clinically reviewed. They are not claimed to come from the uploaded multimodal dataset.

### Planned model interface

`Static valve features -> Valve Encoder (MLP)`  
`Longitudinal physiology/hemodynamics -> Physiology Encoder (GRU)`  
`Longitudinal medications -> Medication Encoder (GRU)`  
`Temporal fusion + valve embedding -> Survival Head -> S(t)`


In [1]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Put the reference CSV in the same folder as this notebook,
# or change this path.
REFERENCE_CSV = Path("multimodal_patient_year.csv")

# Optional schema produced by Step 1.
SCHEMA_CSV = Path("synthetic_design_outputs/03_proposed_synthetic_schema.csv")

OUTDIR = Path("synthetic_generator_outputs")
OUTDIR.mkdir(parents=True, exist_ok=True)

# Default synthetic cohort size.
N_PATIENTS = 1000

# Annual timepoints from implant to 10 years.
TIMEPOINT_MONTHS = np.arange(0, 121, 12)

print("Reference:", REFERENCE_CSV.resolve())
print("Outputs:", OUTDIR.resolve())


Reference: C:\Users\User\OneDrive\Υπολογιστής\πανεπιστημιο\workstation\docathon\multimodal_patient_year.csv
Outputs: C:\Users\User\OneDrive\Υπολογιστής\πανεπιστημιο\workstation\docathon\synthetic_generator_outputs


## 1. Load the reference multimodal dataset

We use it to estimate empirical distributions and missingness where possible.


In [2]:
if not REFERENCE_CSV.exists():
    raise FileNotFoundError(
        f"Reference CSV not found: {REFERENCE_CSV.resolve()}\n"
        "Place multimodal_patient_year.csv next to this notebook or update REFERENCE_CSV."
    )

ref = pd.read_csv(REFERENCE_CSV)

print("Reference shape:", ref.shape)
print("Reference patients:", ref["Patient"].nunique() if "Patient" in ref.columns else "unknown")
display(ref.head())


Reference shape: (226, 82)
Reference patients: 117


,Patient,Year,note_count,note_types,note_services,signed_statuses,provider_types,provider_specialties,min_creation_year,max_creation_year,...,med__antiarrhythmic__present,med__insulin__present,medications_observed,Index_Date,Valve_Failure,notes_available,labs_available,medications_available,rich_lab_med,relative_to_index_year_UNVERIFIED
0,Patient_001,2022,1.0,Operative Report,Cardiac Surgery,Signed,Physician,Cardiac Surg,2022.0,2022.0,...,NaN,NaN,0,2022.0,1.0,1.0,0.0,0.0,0.0,0.0
1,Patient_001,2025,1.0,Progress Notes,NaN,Signed,Physician,Cardiology,2025.0,2025.0,...,NaN,NaN,0,2022.0,1.0,1.0,0.0,0.0,0.0,3.0
2,Patient_002,2017,1.0,Operative Report,Cardiac Surgery,Signed,Physician,NaN,2017.0,2017.0,...,NaN,NaN,0,2018.0,1.0,1.0,0.0,0.0,0.0,-1.0
3,Patient_002,2018,1.0,Progress Notes,NaN,Signed,Physician Assistant,NaN,2018.0,2018.0,...,NaN,NaN,0,2018.0,1.0,1.0,0.0,0.0,0.0,0.0
4,Patient_003,2014,1.0,Operative Report,Cardiac Surgery,Signed,Physician,NaN,2014.0,2014.0,...,NaN,NaN,0,2014.0,0.0,1.0,0.0,0.0,0.0,0.0


## 2. Identify V1 branches

Raw `note_text` is **not** used in V1.

Potential post-event leakage variables such as `Valve_Failure`, `note_mentions_prosthetic_failure`,
`note_mentions_redo`, and `note_mentions_valve_in_valve` are not used as encoder inputs.


In [3]:
LAB_COLS = [c for c in ref.columns if c.startswith("lab__")]
MED_COLS = [c for c in ref.columns if c.startswith("med__") and c.endswith("__present")]

LEAKAGE_NOTE_COLS = {
    "note_mentions_prosthetic_failure",
    "note_mentions_redo",
    "note_mentions_valve_in_valve",
}

SAFE_NOTE_COLS = [
    c for c in ref.columns
    if c.startswith("note_mentions_") and c not in LEAKAGE_NOTE_COLS
]

print("Lab features:", len(LAB_COLS))
print("Medication indicators:", len(MED_COLS))
print("Safe note-derived features:", SAFE_NOTE_COLS)


Lab features: 30
Medication indicators: 15
Safe note-derived features: ['note_mentions_tavr', 'note_mentions_savr_or_avr', 'note_mentions_endocarditis', 'note_mentions_bioprosthetic', 'note_mentions_mechanical', 'note_mentions_cabg']


## 3. Empirical reference statistics

For numeric labs, we estimate median/IQR and observed rate.

For medication indicators, we estimate prevalence among observed values.

These empirical values help the synthetic data retain some of the structure of the reference dataset.


In [4]:
def numeric_reference_stats(df, columns):
    rows = []
    for c in columns:
        s = pd.to_numeric(df[c], errors="coerce")
        observed = s.dropna()
        if observed.empty:
            continue
        q25, med, q75 = observed.quantile([0.25, 0.5, 0.75])
        rows.append({
            "feature": c,
            "median": float(med),
            "q25": float(q25),
            "q75": float(q75),
            "iqr": float(q75 - q25),
            "min": float(observed.min()),
            "max": float(observed.max()),
            "observed_rate": float(s.notna().mean()),
        })
    return pd.DataFrame(rows)

lab_stats = numeric_reference_stats(ref, LAB_COLS)

med_stats = []
for c in MED_COLS:
    s = pd.to_numeric(ref[c], errors="coerce")
    observed = s.dropna()
    if observed.empty:
        continue
    med_stats.append({
        "feature": c,
        "prevalence": float(observed.mean()),
        "observed_rate": float(s.notna().mean()),
    })
med_stats = pd.DataFrame(med_stats)

display(lab_stats.head(20))
display(med_stats.head(20))


,feature,median,q25,q75,iqr,min,max,observed_rate
0,lab__ALT,19.000,15.0000,31.87500,16.87500,8.00,139.00,0.168142
1,lab__APTT,30.200,28.2000,31.20000,3.00000,25.60,71.40,0.092920
2,lab__AST,23.000,18.3750,28.00000,9.62500,13.00,121.00,0.168142
3,lab__Albumin,3.725,3.1000,4.10000,1.00000,2.60,4.60,0.168142
4,lab__Alkaline Phosphatase,62.500,50.2500,79.50000,29.25000,33.00,114.50,0.168142
5,lab__Anion Gap,11.000,10.0000,12.00000,2.00000,7.00,16.50,0.194690
6,lab__BUN,20.000,17.0000,24.00000,7.00000,4.00,93.00,0.194690
7,lab__CO2,26.000,25.0000,27.00000,2.00000,22.00,31.00,0.194690
8,lab__Calcium,8.950,8.6750,9.46250,0.78750,8.10,10.60,0.194690
9,lab__Chloride,102.000,101.0000,104.00000,3.00000,93.00,106.00,0.194690


,feature,prevalence,observed_rate
0,med__beta_blocker__present,0.4750,0.353982
1,med__ace_inhibitor__present,0.2375,0.353982
2,med__arb__present,0.2250,0.353982
3,med__arni__present,0.1125,0.353982
4,med__loop_diuretic__present,0.3500,0.353982
5,med__thiazide_diuretic__present,0.1125,0.353982
6,med__mra__present,0.0750,0.353982
7,med__anticoagulant__present,0.3625,0.353982
8,med__antiplatelet__present,0.3750,0.353982
9,med__statin__present,0.4375,0.353982


## 4. Generator design

We use four hidden synthetic phenotypes:

- `stable`
- `slow_deterioration`
- `rapid_deterioration`
- `regurgitation_dominant`

The phenotype is used **only by the generator**.  
It must **not** be passed to the predictive model.

The valve-specific values below are synthetic assumptions and should be clinician-reviewed before being treated as realistic.


In [5]:
PHENOTYPE_PROBS = {
    "stable": 0.55,
    "slow_deterioration": 0.25,
    "rapid_deterioration": 0.10,
    "regurgitation_dominant": 0.10,
}

# Synthetic static valve distributions.
PROCEDURE_TYPES = ["TAVR", "SAVR"]
PROCEDURE_PROBS = [0.65, 0.35]

VALVE_MODELS = ["Synthetic_Model_A", "Synthetic_Model_B", "Synthetic_Model_C"]
VALVE_MODEL_PROBS = [0.40, 0.35, 0.25]

VALVE_SIZES_MM = [19, 21, 23, 25, 26, 27, 29]
VALVE_SIZE_PROBS = np.array([0.05, 0.12, 0.20, 0.18, 0.18, 0.15, 0.12])
VALVE_SIZE_PROBS = VALVE_SIZE_PROBS / VALVE_SIZE_PROBS.sum()

# Illustrative event-time distributions in months.
EVENT_TIME_RULES = {
    "stable":                {"event_prob": 0.08, "mean": 108, "sd": 18},
    "slow_deterioration":    {"event_prob": 0.55, "mean": 84,  "sd": 18},
    "rapid_deterioration":   {"event_prob": 0.85, "mean": 48,  "sd": 14},
    "regurgitation_dominant":{"event_prob": 0.60, "mean": 72,  "sd": 18},
}

# Baseline synthetic valve hemodynamics (illustrative, clinician-review required).
BASELINE_HEMO = {
    "mean_gradient_mmHg": (10.0, 2.5),
    "peak_velocity_m_s": (2.0, 0.25),
    "effective_orifice_area_cm2": (1.8, 0.25),
    "regurgitation_grade": (0.4, 0.5),  # ordinal-ish continuous before rounding
}


## 5. Helper functions

The generator is built in layers:

1. static patient/valve profile,
2. event/censor time,
3. longitudinal hemodynamics,
4. longitudinal labs,
5. medications,
6. note-derived signals,
7. missingness.


In [6]:
def clipped_normal(mean, sd, low=None, high=None, size=None):
    x = rng.normal(mean, sd, size=size)
    if low is not None:
        x = np.maximum(x, low)
    if high is not None:
        x = np.minimum(x, high)
    return x


def sample_phenotype():
    names = list(PHENOTYPE_PROBS.keys())
    probs = list(PHENOTYPE_PROBS.values())
    return rng.choice(names, p=probs)


def generate_event_and_censor(phenotype):
    rule = EVENT_TIME_RULES[phenotype]
    has_event = rng.random() < rule["event_prob"]

    event_time = None
    if has_event:
        event_time = float(clipped_normal(rule["mean"], rule["sd"], low=18, high=120))

    # Administrative/random censoring.
    censor_time = float(rng.uniform(48, 120))

    if event_time is not None and event_time <= censor_time:
        return 1, event_time, censor_time

    return 0, None, censor_time


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def hemodynamic_trajectory(phenotype, months, baseline, event_time=None):
    t_years = months / 12.0

    grad0 = baseline["mean_gradient_mmHg"]
    vel0 = baseline["peak_velocity_m_s"]
    eoa0 = baseline["effective_orifice_area_cm2"]
    reg0 = baseline["regurgitation_grade"]

    # Slow age-related drift for all patients.
    grad = grad0 + 0.35 * t_years
    vel = vel0 + 0.025 * t_years
    eoa = eoa0 - 0.015 * t_years
    reg = reg0 + 0.03 * t_years

    if phenotype == "slow_deterioration":
        grad += 0.9 * np.maximum(t_years - 2, 0) ** 1.35
        vel += 0.07 * np.maximum(t_years - 2, 0) ** 1.20
        eoa -= 0.055 * np.maximum(t_years - 2, 0) ** 1.25

    elif phenotype == "rapid_deterioration":
        grad += 2.0 * np.maximum(t_years - 1, 0) ** 1.35
        vel += 0.14 * np.maximum(t_years - 1, 0) ** 1.20
        eoa -= 0.10 * np.maximum(t_years - 1, 0) ** 1.20

    elif phenotype == "regurgitation_dominant":
        reg += 0.42 * np.maximum(t_years - 2, 0) ** 1.15
        grad += 0.25 * np.maximum(t_years - 3, 0)
        eoa -= 0.015 * np.maximum(t_years - 3, 0)

    # Add modest patient-time noise.
    grad += rng.normal(0, 1.2)
    vel += rng.normal(0, 0.08)
    eoa += rng.normal(0, 0.08)
    reg += rng.normal(0, 0.20)

    grad = float(np.clip(grad, 3, 60))
    vel = float(np.clip(vel, 0.8, 6.0))
    eoa = float(np.clip(eoa, 0.35, 3.0))
    reg_grade = int(np.clip(np.rint(reg), 0, 4))

    return grad, vel, eoa, reg_grade


def sample_lab_value(feature, phenotype, months, hf_burden=0.0):
    row = lab_stats[lab_stats["feature"] == feature]
    if row.empty:
        return np.nan

    row = row.iloc[0]
    median = row["median"]
    iqr = row["iqr"]

    # Robust scale; avoid zero variance.
    scale = max(iqr / 1.349, abs(median) * 0.05, 1e-6)
    value = rng.normal(median, scale)

    # Gentle longitudinal disease-linked shifts for selected commonly useful labs.
    name = feature.lower()
    t_years = months / 12.0

    if "nt-probnp" in name or "bnp" in name:
        value = max(0, value * (1 + 0.10 * hf_burden + 0.03 * t_years))
    elif "creatinine" in name:
        value = max(0, value * (1 + 0.03 * hf_burden + 0.01 * t_years))
    elif "egfr" in name:
        value = max(1, value * (1 - 0.025 * hf_burden - 0.01 * t_years))
    elif "lvef" in name:
        value = value - 1.2 * hf_burden - 0.25 * t_years

    # Keep within empirical min/max where possible.
    value = float(np.clip(value, row["min"], row["max"]))
    return value


def medication_probability(feature, hf_burden, af, ckd):
    row = med_stats[med_stats["feature"] == feature]
    base = float(row.iloc[0]["prevalence"]) if not row.empty else 0.20

    name = feature.lower()
    logit = math.log((base + 1e-4) / (1 - base + 1e-4))

    if "loop_diuretic" in name:
        logit += 0.9 * hf_burden
    if "anticoagulant" in name:
        logit += 1.5 * af
    if "sglt2" in name:
        logit += 0.35 * hf_burden + 0.20 * ckd
    if "beta_blocker" in name:
        logit += 0.25 * hf_burden + 0.25 * af

    return float(np.clip(sigmoid(logit), 0.01, 0.99))


## 6. Generate synthetic patients

We generate one row per **patient-timepoint**.

Static variables repeat across a patient's rows.  
Dynamic variables evolve over time.


In [7]:
rows = []

for i in range(N_PATIENTS):
    patient_id = f"SYN_{i+1:05d}"
    phenotype = sample_phenotype()

    procedure_type = rng.choice(PROCEDURE_TYPES, p=PROCEDURE_PROBS)
    valve_model = rng.choice(VALVE_MODELS, p=VALVE_MODEL_PROBS)
    valve_size_mm = int(rng.choice(VALVE_SIZES_MM, p=VALVE_SIZE_PROBS))
    valve_position = "aortic"

    age_at_implant = int(np.clip(rng.normal(76 if procedure_type == "TAVR" else 68, 8), 45, 95))
    sex = rng.choice(["Female", "Male"], p=[0.46, 0.54])

    # Simple synthetic comorbidities.
    ckd = int(rng.random() < np.clip(0.18 + 0.006 * (age_at_implant - 60), 0.05, 0.55))
    af = int(rng.random() < np.clip(0.20 + 0.004 * (age_at_implant - 60), 0.08, 0.50))
    heart_failure = int(rng.random() < np.clip(0.18 + 0.10 * (phenotype != "stable"), 0.10, 0.55))
    diabetes = int(rng.random() < 0.28)

    event, event_time, censor_time = generate_event_and_censor(phenotype)
    followup_end = event_time if event == 1 else censor_time

    baseline = {
        "mean_gradient_mmHg": float(clipped_normal(*BASELINE_HEMO["mean_gradient_mmHg"], low=4, high=20)),
        "peak_velocity_m_s": float(clipped_normal(*BASELINE_HEMO["peak_velocity_m_s"], low=1.2, high=3.0)),
        "effective_orifice_area_cm2": float(clipped_normal(*BASELINE_HEMO["effective_orifice_area_cm2"], low=1.0, high=2.8)),
        "regurgitation_grade": float(clipped_normal(*BASELINE_HEMO["regurgitation_grade"], low=0, high=2)),
    }

    for months in TIMEPOINT_MONTHS:
        if months > followup_end:
            continue

        grad, vel, eoa, reg_grade = hemodynamic_trajectory(
            phenotype, months, baseline, event_time=event_time
        )

        # Latent clinical burden derived from current state, not from event label.
        stenotic_burden = np.clip((grad - 12) / 20, 0, 2)
        regurg_burden = reg_grade / 3.0
        hf_burden = float(np.clip(
            0.45 * heart_failure + 0.55 * stenotic_burden + 0.65 * regurg_burden,
            0, 3
        ))

        row = {
            "Patient": patient_id,
            "time_since_implant_months": int(months),
            "procedure_type": procedure_type,
            "valve_model": valve_model,
            "valve_size_mm": valve_size_mm,
            "valve_position": valve_position,
            "age_at_implant": age_at_implant,
            "sex": sex,
            "ckd": ckd,
            "af": af,
            "heart_failure": heart_failure,
            "diabetes": diabetes,

            "mean_gradient_mmHg": grad,
            "peak_velocity_m_s": vel,
            "effective_orifice_area_cm2": eoa,
            "regurgitation_grade": reg_grade,

            # Generator-only field. Drop before model training.
            "generator_latent_phenotype": phenotype,

            # Survival targets repeated per patient for convenience.
            "duration_months": float(followup_end),
            "event": int(event),
            "event_type": (
                "regurgitation_dominant" if (event == 1 and phenotype == "regurgitation_dominant")
                else "hemodynamic_deterioration" if event == 1
                else "censored"
            ),
        }

        # Reference-based lab generation.
        for lab in LAB_COLS:
            row[lab] = sample_lab_value(lab, phenotype, months, hf_burden=hf_burden)

        # Medication indicators conditioned on current burden/comorbidities.
        for med in MED_COLS:
            p = medication_probability(med, hf_burden=hf_burden, af=af, ckd=ckd)
            row[med] = int(rng.random() < p)

        # V1 structured note-derived signals.
        # These reflect the CURRENT state only.
        row["note_dyspnea"] = int(rng.random() < sigmoid(-2.2 + 1.2 * hf_burden))
        row["note_valve_dysfunction"] = int(
            rng.random() < sigmoid(-3.0 + 0.11 * max(grad - 10, 0) + 0.7 * reg_grade)
        )

        # Explicitly create leakage columns only for auditing realism.
        # They must NOT be encoder inputs.
        occurred_by_now = int(event == 1 and event_time is not None and months >= event_time)
        row["note_mentions_prosthetic_failure"] = occurred_by_now
        row["Valve_Failure"] = occurred_by_now

        rows.append(row)

synthetic = pd.DataFrame(rows)

print("Synthetic shape:", synthetic.shape)
print("Synthetic patients:", synthetic["Patient"].nunique())
display(synthetic.head())


Synthetic shape: (7010, 69)
Synthetic patients: 1000


,Patient,time_since_implant_months,procedure_type,valve_model,valve_size_mm,valve_position,age_at_implant,sex,ckd,af,...,med__statin__present,med__ccb__present,med__sglt2_inhibitor__present,med__digoxin__present,med__antiarrhythmic__present,med__insulin__present,note_dyspnea,note_valve_dysfunction,note_mentions_prosthetic_failure,Valve_Failure
0,SYN_00001,0,TAVR,Synthetic_Model_C,26,aortic,60,Male,0,0,...,0,0,0,0,0,1,1,0,0,0
1,SYN_00001,12,TAVR,Synthetic_Model_C,26,aortic,60,Male,0,0,...,0,1,0,0,0,0,0,0,0,0
2,SYN_00001,24,TAVR,Synthetic_Model_C,26,aortic,60,Male,0,0,...,0,0,0,0,0,0,0,0,0,0
3,SYN_00001,36,TAVR,Synthetic_Model_C,26,aortic,60,Male,0,0,...,1,1,0,0,0,0,0,0,0,0
4,SYN_00001,48,TAVR,Synthetic_Model_C,26,aortic,60,Male,0,0,...,1,1,1,0,0,0,0,0,0,0


## 7. Apply realistic missingness

We approximate the reference dataset's observed rates for labs and medication features.

This is intentionally simple in V1.  
Later we can make missingness depend on severity, visit type, or time.


In [8]:
synthetic_missing = synthetic.copy()

# Labs: use reference column-level observed rates.
for _, r in lab_stats.iterrows():
    c = r["feature"]
    observed_rate = float(r["observed_rate"])
    keep_mask = rng.random(len(synthetic_missing)) < observed_rate
    synthetic_missing.loc[~keep_mask, c] = np.nan

# Medications: use reference column-level observed rates.
for _, r in med_stats.iterrows():
    c = r["feature"]
    observed_rate = float(r["observed_rate"])
    keep_mask = rng.random(len(synthetic_missing)) < observed_rate
    synthetic_missing.loc[~keep_mask, c] = np.nan

print("Missingness applied.")
display(
    synthetic_missing[
        [c for c in LAB_COLS[:5] if c in synthetic_missing.columns] +
        [c for c in MED_COLS[:5] if c in synthetic_missing.columns]
    ].isna().mean().to_frame("synthetic_missing_rate")
)


Missingness applied.


,synthetic_missing_rate
lab__ALT,0.829957
lab__APTT,0.909986
lab__AST,0.832953
lab__Albumin,0.831954
lab__Alkaline Phosphatase,0.829957
med__beta_blocker__present,0.647218
med__ace_inhibitor__present,0.655064
med__arb__present,0.650357
med__arni__present,0.649786
med__loop_diuretic__present,0.655492


## 8. Save the synthetic dataset

Two versions are saved:

- **full**: includes generator/debug fields and leakage audit variables,
- **model-ready**: removes the latent phenotype and explicit leakage columns.

The model-ready file is the one that should later feed the encoder pipeline.


In [9]:
full_path = OUTDIR / "synthetic_multimodal_patient_year_v1_FULL.csv"
model_path = OUTDIR / "synthetic_multimodal_patient_year_v1_MODEL_READY.csv"

synthetic_missing.to_csv(full_path, index=False)

DROP_FROM_MODEL = [
    "generator_latent_phenotype",
    "Valve_Failure",
    "note_mentions_prosthetic_failure",
]

model_ready = synthetic_missing.drop(
    columns=[c for c in DROP_FROM_MODEL if c in synthetic_missing.columns]
).copy()

model_ready.to_csv(model_path, index=False)

print("Saved full dataset:", full_path.resolve())
print("Saved model-ready dataset:", model_path.resolve())
print("Model-ready shape:", model_ready.shape)


Saved full dataset: C:\Users\User\OneDrive\Υπολογιστής\πανεπιστημιο\workstation\docathon\synthetic_generator_outputs\synthetic_multimodal_patient_year_v1_FULL.csv
Saved model-ready dataset: C:\Users\User\OneDrive\Υπολογιστής\πανεπιστημιο\workstation\docathon\synthetic_generator_outputs\synthetic_multimodal_patient_year_v1_MODEL_READY.csv
Model-ready shape: (7010, 66)


## 9. Define exact V1 encoder inputs

This cell creates the feature lists that the next notebook will use.

No encoder is trained here.


In [10]:
STATIC_FEATURES = [
    "procedure_type",
    "valve_model",
    "valve_size_mm",
    "valve_position",
    "age_at_implant",
    "sex",
    "ckd",
    "af",
    "heart_failure",
    "diabetes",
]

PHYSIOLOGY_FEATURES = [
    "mean_gradient_mmHg",
    "peak_velocity_m_s",
    "effective_orifice_area_cm2",
    "regurgitation_grade",
] + LAB_COLS

MEDICATION_FEATURES = MED_COLS

NOTE_DERIVED_FEATURES = [
    "note_dyspnea",
    "note_valve_dysfunction",
]

TARGET_COLUMNS = [
    "duration_months",
    "event",
    "event_type",
]

encoder_spec = {
    "static_features": STATIC_FEATURES,
    "physiology_features": PHYSIOLOGY_FEATURES,
    "medication_features": MEDICATION_FEATURES,
    "note_derived_features_v1_optional": NOTE_DERIVED_FEATURES,
    "targets": TARGET_COLUMNS,
    "time_column": "time_since_implant_months",
    "patient_id_column": "Patient",
}

with open(OUTDIR / "encoder_input_spec_v1.json", "w", encoding="utf-8") as f:
    json.dump(encoder_spec, f, indent=2, ensure_ascii=False)

print(json.dumps(encoder_spec, indent=2, ensure_ascii=False))


{
  "static_features": [
    "procedure_type",
    "valve_model",
    "valve_size_mm",
    "valve_position",
    "age_at_implant",
    "sex",
    "ckd",
    "af",
    "heart_failure",
    "diabetes"
  ],
  "physiology_features": [
    "mean_gradient_mmHg",
    "peak_velocity_m_s",
    "effective_orifice_area_cm2",
    "regurgitation_grade",
    "lab__ALT",
    "lab__APTT",
    "lab__AST",
    "lab__Albumin",
    "lab__Alkaline Phosphatase",
    "lab__Anion Gap",
    "lab__BUN",
    "lab__CO2",
    "lab__Calcium",
    "lab__Chloride",
    "lab__Creatinine",
    "lab__Glucose",
    "lab__Hematocrit",
    "lab__Hemoglobin",
    "lab__INR",
    "lab__LVEF",
    "lab__MCH",
    "lab__MCHC",
    "lab__MCV",
    "lab__NT-proBNP",
    "lab__Platelets",
    "lab__Potassium",
    "lab__RBC",
    "lab__RDW",
    "lab__Sodium",
    "lab__Total Bilirubin",
    "lab__Total Protein",
    "lab__Troponin T",
    "lab__WBC",
    "lab__eGFR"
  ],
  "medication_features": [
    "med__beta_blocker__present

## 10. Sanity checks

These checks do not prove clinical validity.  
They only verify that the synthetic dataset behaves as intended structurally.


In [11]:
checks = {
    "patients": int(model_ready["Patient"].nunique()),
    "rows": int(len(model_ready)),
    "event_rate_patient_level": float(
        model_ready.groupby("Patient")["event"].first().mean()
    ),
    "median_followup_months": float(
        model_ready.groupby("Patient")["duration_months"].first().median()
    ),
    "median_timepoints_per_patient": float(
        model_ready.groupby("Patient").size().median()
    ),
}

print(json.dumps(checks, indent=2))

# Inspect phenotype behavior using FULL dataset only (generator QA).
qa = (
    synthetic_missing.groupby("generator_latent_phenotype")
    .agg(
        patients=("Patient", "nunique"),
        event_rate=("event", "mean"),
        mean_gradient=("mean_gradient_mmHg", "mean"),
        mean_regurgitation=("regurgitation_grade", "mean"),
    )
)
display(qa)


{
  "patients": 1000,
  "rows": 7010,
  "event_rate_patient_level": 0.187,
  "median_followup_months": 75.78295828724696,
  "median_timepoints_per_patient": 7.0
}


,patients,event_rate,mean_gradient,mean_regurgitation
generator_latent_phenotype,,,,
rapid_deterioration,100,0.730612,14.982241,0.504082
regurgitation_dominant,93,0.250000,11.384839,1.333841
slow_deterioration,257,0.274302,13.451041,0.523464
stable,550,0.011782,11.148520,0.540010


# What we do next

After reviewing the generated dataset:

1. verify distributions and missingness,
2. adjust valve-specific assumptions with clinical input,
3. confirm which note-derived variables stay in V1,
4. then build **Step 3: tensor preparation + encoder architecture**.

The next notebook should create:

- `X_static`
- `X_physiology [N, T, F_phys]`
- `X_medications [N, T, F_med]`
- `mask [N, T]`
- `duration [N]`
- `event [N]`

and only then instantiate:

**Valve MLP + Physiology GRU + Medication GRU + Fusion + Survival Head**.
